# Notebook 04 — Feature Engineering

**Project:** Financial Fraud Detection  
**Author:** Sarva  
**Date:** September 2026  

---

## Objective

Create meaningful, fraud-detection-oriented features from the cleaned dataset **without introducing data leakage**.

### Scope
| Area | Details |
|---|---|
| **Input** | `data/processed/cleaned_fraud_dataset.csv` |
| **Output** | `data/processed/feature_engineered_fraud_dataset.csv` |
| **Feature Groups** | Temporal · Transaction · Customer Behavioral |
| **Excluded Steps** | Train/test split, SMOTE, scaling, modelling |

> **Leakage policy:** The target column `Fraud_Label` is **never** used to derive any engineered feature.

---
## 1 · Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")

pandas  : 3.0.5
numpy   : 2.4.6


---
## 2 · Load the Cleaned Dataset

A dynamic path is constructed relative to this notebook so the code works on any operating system.

In [2]:
# Build path relative to the notebook location
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, os.pardir))

INPUT_PATH  = os.path.join(PROJECT_ROOT, "data", "processed", "cleaned_fraud_dataset.csv")
OUTPUT_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "feature_engineered_fraud_dataset.csv")

print(f"Input  : {INPUT_PATH}")
print(f"Output : {OUTPUT_PATH}")

Input  : c:\Users\sarva\Desktop\financial-fraud-detection\data\processed\cleaned_fraud_dataset.csv
Output : c:\Users\sarva\Desktop\financial-fraud-detection\data\processed\feature_engineered_fraud_dataset.csv


In [3]:
df = pd.read_csv(INPUT_PATH, parse_dates=["Date"])
print(f"Dataset loaded — {df.shape[0]:,} rows × {df.shape[1]} columns")

Dataset loaded — 50,000 rows × 14 columns


---
## 3 · Verify Shape, Columns & Data Types

In [4]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (50000, 14)

Columns:
['Transaction_ID', 'User_ID', 'Transaction_Amount', 'Transaction_Type', 'Date', 'Account_Balance', 'Device_Type', 'Location', 'Merchant_Category', 'Previous_Fraudulent_Activity', 'Daily_Transaction_Count', 'Card_Type', 'Card_Age', 'Fraud_Label']


In [5]:
df.dtypes

Transaction_ID                             str
User_ID                                    str
Transaction_Amount                     float64
Transaction_Type                           str
Date                            datetime64[us]
Account_Balance                        float64
Device_Type                                str
Location                                   str
Merchant_Category                          str
Previous_Fraudulent_Activity             int64
Daily_Transaction_Count                  int64
Card_Type                                  str
Card_Age                                 int64
Fraud_Label                              int64
dtype: object

In [6]:
df.head(3)

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Date,Account_Balance,Device_Type,Location,Merchant_Category,Previous_Fraudulent_Activity,Daily_Transaction_Count,Card_Type,Card_Age,Fraud_Label
0,TXN_33553,USER_1834,39.79,POS,2023-08-14,93213.17,Laptop,Sydney,Travel,0,7,Amex,65,0
1,TXN_9427,USER_7875,1.19,Bank Transfer,2023-06-07,75725.25,Mobile,New York,Clothing,0,13,Mastercard,186,1
2,TXN_199,USER_2734,28.96,Online,2023-06-20,1588.96,Tablet,Mumbai,Restaurants,0,14,Visa,226,1


In [7]:
# Store original row count for later validation
ORIGINAL_ROW_COUNT = df.shape[0]
ORIGINAL_COLUMNS  = df.columns.tolist()

print(f"Original row count : {ORIGINAL_ROW_COUNT:,}")
print(f"Original col count : {len(ORIGINAL_COLUMNS)}")

Original row count : 50,000
Original col count : 14


---
## 4 · Identify Columns by Role

Before engineering features, let's categorize each column so the intent is clear.

| Role | Columns |
|---|---|
| **Identifiers** (not model features) | `Transaction_ID`, `User_ID` |
| **Target** | `Fraud_Label` |
| **Features** | Everything else |

In [8]:
ID_COLUMNS     = ["Transaction_ID", "User_ID"]
TARGET_COLUMN  = "Fraud_Label"
FEATURE_COLUMNS = [c for c in df.columns if c not in ID_COLUMNS + [TARGET_COLUMN]]

print("Identifier columns :", ID_COLUMNS)
print("Target column       :", TARGET_COLUMN)
print("Feature columns     :", FEATURE_COLUMNS)

Identifier columns : ['Transaction_ID', 'User_ID']
Target column       : Fraud_Label
Feature columns     : ['Transaction_Amount', 'Transaction_Type', 'Date', 'Account_Balance', 'Device_Type', 'Location', 'Merchant_Category', 'Previous_Fraudulent_Activity', 'Daily_Transaction_Count', 'Card_Type', 'Card_Age']


---
## 5 · Temporal Features

Fraud patterns often vary by **time of year**, **day of week**, and **day of month**.  
We extract granular temporal components from the `Date` column.

In [9]:
# Ensure Date is datetime
df["Date"] = pd.to_datetime(df["Date"])

In [10]:
# 5.1 — Year
df["Year"] = df["Date"].dt.year

# 5.2 — Month (1-12)
df["Month"] = df["Date"].dt.month

# 5.3 — Day of month (1-31)
df["Day"] = df["Date"].dt.day

# 5.4 — Day of week (0 = Monday … 6 = Sunday)
df["Day_of_Week"] = df["Date"].dt.dayofweek

print("Temporal features created: Year, Month, Day, Day_of_Week")
df[["Date", "Year", "Month", "Day", "Day_of_Week"]].head(5)

Temporal features created: Year, Month, Day, Day_of_Week


,Date,Year,Month,Day,Day_of_Week
0,2023-08-14,2023,8,14,0
1,2023-06-07,2023,6,7,2
2,2023-06-20,2023,6,20,1
3,2023-12-07,2023,12,7,3
4,2023-11-11,2023,11,11,5


In [11]:
# Quick sanity check
print("Year  unique values :", sorted(df["Year"].unique()))
print("Month range         :", df["Month"].min(), "–", df["Month"].max())
print("Day   range         :", df["Day"].min(), "–", df["Day"].max())
print("Day_of_Week range   :", df["Day_of_Week"].min(), "–", df["Day_of_Week"].max())

Year  unique values : [np.int32(2023)]
Month range         : 1 – 12
Day   range         : 1 – 31
Day_of_Week range   : 0 – 6


---
## 6 · Transaction Features

### 6.1 — Amount Bins

Binning `Transaction_Amount` into ordinal categories helps the model capture **non-linear amount thresholds** common in fraud (e.g., micro-transactions used for card testing).

In [12]:
# Define bin edges based on domain knowledge and data distribution
amount_bins   = [0, 10, 50, 100, 250, 500, np.inf]
amount_labels = ["micro", "low", "medium", "high", "very_high", "extreme"]

df["Amount_Bin"] = pd.cut(
    df["Transaction_Amount"],
    bins=amount_bins,
    labels=amount_labels,
    include_lowest=True
)

print("Amount_Bin distribution:")
print(df["Amount_Bin"].value_counts().sort_index())

Amount_Bin distribution:
Amount_Bin
micro         4788
low          14878
medium       11869
high         14489
very_high     3681
extreme        295
Name: count, dtype: int64


### 6.2 — Balance-to-Amount Ratio

A low `Account_Balance / Transaction_Amount` ratio may signal suspicious activity — the transaction is unusually large relative to the account balance.

In [13]:
# Avoid division by zero: where amount is 0, ratio is set to NaN then filled
df["Balance_to_Amount_Ratio"] = np.where(
    df["Transaction_Amount"] == 0,
    np.nan,
    df["Account_Balance"] / df["Transaction_Amount"]
)

# Fill any NaN ratios with the column median (safe, no leakage)
median_ratio = df["Balance_to_Amount_Ratio"].median()
df["Balance_to_Amount_Ratio"] = df["Balance_to_Amount_Ratio"].fillna(median_ratio)

print(f"Median ratio used for fill: {median_ratio:.2f}")
print("\nBalance_to_Amount_Ratio statistics:")
print(df["Balance_to_Amount_Ratio"].describe())

Median ratio used for fill: 633.71

Balance_to_Amount_Ratio statistics:
count    5.000000e+04
mean     4.916037e+03
std      8.993721e+04
min      9.504424e-01
25%      2.622435e+02
50%      6.337057e+02
75%      1.675386e+03
max      9.784693e+06
Name: Balance_to_Amount_Ratio, dtype: float64


---
## 7 · Customer Behavioral Features (User-Level Aggregations)

Aggregating transaction behavior **per user** captures patterns that single-row features cannot.  
These are computed using only non-target columns (`Transaction_Amount`).

| Feature | Description |
|---|---|
| `User_Transaction_Count` | Total number of transactions for the user |
| `User_Avg_Transaction_Amount` | Mean transaction amount per user |
| `User_Total_Transaction_Amount` | Sum of all transaction amounts per user |
| `User_Amount_Deviation` | How far each transaction is from the user's mean |

In [14]:
# 7.1 — User Transaction Count
user_txn_count = df.groupby("User_ID")["Transaction_Amount"].transform("count")
df["User_Transaction_Count"] = user_txn_count

print("User_Transaction_Count statistics:")
print(df["User_Transaction_Count"].describe())

User_Transaction_Count statistics:
count    50000.000000
mean         6.553120
std          2.358281
min          1.000000
25%          5.000000
50%          6.000000
75%          8.000000
max         16.000000
Name: User_Transaction_Count, dtype: float64


In [15]:
# 7.2 — User Average Transaction Amount
df["User_Avg_Transaction_Amount"] = df.groupby("User_ID")["Transaction_Amount"].transform("mean")

print("User_Avg_Transaction_Amount statistics:")
print(df["User_Avg_Transaction_Amount"].describe())

User_Avg_Transaction_Amount statistics:
count    50000.000000
mean        99.411012
std         41.849256
min          0.040000
25%         70.002500
50%         94.184286
75%        121.800000
max        523.280000
Name: User_Avg_Transaction_Amount, dtype: float64


In [16]:
# 7.3 — User Total Transaction Amount
df["User_Total_Transaction_Amount"] = df.groupby("User_ID")["Transaction_Amount"].transform("sum")

print("User_Total_Transaction_Amount statistics:")
print(df["User_Total_Transaction_Amount"].describe())

User_Total_Transaction_Amount statistics:
count    50000.00000
mean       651.65848
std        346.33074
min          0.04000
25%        398.26000
50%        602.92000
75%        845.57000
max       2395.02000
Name: User_Total_Transaction_Amount, dtype: float64


In [17]:
# 7.4 — User Amount Deviation
# Measures how much each transaction deviates from the user's average.
# Large deviations may indicate anomalous (potentially fraudulent) behavior.
df["User_Amount_Deviation"] = (
    df["Transaction_Amount"] - df["User_Avg_Transaction_Amount"]
)

print("User_Amount_Deviation statistics:")
print(df["User_Amount_Deviation"].describe())

User_Amount_Deviation statistics:
count    5.000000e+04
mean    -7.275958e-17
std      8.937461e+01
min     -3.731350e+02
25%     -5.704700e+01
50%     -1.673612e+01
75%      3.907458e+01
max      9.116120e+02
Name: User_Amount_Deviation, dtype: float64


---
## 8 · Existing Behavioral Features

The following features already exist in the cleaned dataset and are **retained as-is**:

- `Previous_Fraudulent_Activity` — Whether the user had any prior fraud flags  
- `Daily_Transaction_Count` — Number of transactions on the same day  

No modifications are needed.

In [18]:
print("Previous_Fraudulent_Activity — value counts:")
print(df["Previous_Fraudulent_Activity"].value_counts())
print("\nDaily_Transaction_Count — statistics:")
print(df["Daily_Transaction_Count"].describe())

Previous_Fraudulent_Activity — value counts:
Previous_Fraudulent_Activity
0    45080
1     4920
Name: count, dtype: int64

Daily_Transaction_Count — statistics:
count    50000.000000
mean         7.485240
std          4.039637
min          1.000000
25%          4.000000
50%          7.000000
75%         11.000000
max         14.000000
Name: Daily_Transaction_Count, dtype: float64


---
## 9 · Before / After Column Comparison

In [19]:
NEW_COLUMNS = [c for c in df.columns if c not in ORIGINAL_COLUMNS]

print(f"Original columns ({len(ORIGINAL_COLUMNS)}):")
print(ORIGINAL_COLUMNS)
print(f"\nNew columns ({len(NEW_COLUMNS)}):")
print(NEW_COLUMNS)
print(f"\nFinal columns ({len(df.columns)}):")
print(df.columns.tolist())

Original columns (14):
['Transaction_ID', 'User_ID', 'Transaction_Amount', 'Transaction_Type', 'Date', 'Account_Balance', 'Device_Type', 'Location', 'Merchant_Category', 'Previous_Fraudulent_Activity', 'Daily_Transaction_Count', 'Card_Type', 'Card_Age', 'Fraud_Label']

New columns (10):
['Year', 'Month', 'Day', 'Day_of_Week', 'Amount_Bin', 'Balance_to_Amount_Ratio', 'User_Transaction_Count', 'User_Avg_Transaction_Amount', 'User_Total_Transaction_Amount', 'User_Amount_Deviation']

Final columns (24):
['Transaction_ID', 'User_ID', 'Transaction_Amount', 'Transaction_Type', 'Date', 'Account_Balance', 'Device_Type', 'Location', 'Merchant_Category', 'Previous_Fraudulent_Activity', 'Daily_Transaction_Count', 'Card_Type', 'Card_Age', 'Fraud_Label', 'Year', 'Month', 'Day', 'Day_of_Week', 'Amount_Bin', 'Balance_to_Amount_Ratio', 'User_Transaction_Count', 'User_Avg_Transaction_Amount', 'User_Total_Transaction_Amount', 'User_Amount_Deviation']


---
## 10 · Summary Statistics for New Numerical Features

In [20]:
# Select only the newly created numerical features
new_numeric = [c for c in NEW_COLUMNS if df[c].dtype in ["int64", "int32", "float64"]]

print(f"New numerical features: {new_numeric}\n")
df[new_numeric].describe().round(2)

New numerical features: ['Year', 'Month', 'Day', 'Day_of_Week', 'Balance_to_Amount_Ratio', 'User_Transaction_Count', 'User_Avg_Transaction_Amount', 'User_Total_Transaction_Amount', 'User_Amount_Deviation']



,Year,Month,Day,Day_of_Week,Balance_to_Amount_Ratio,User_Transaction_Count,User_Avg_Transaction_Amount,User_Total_Transaction_Amount,User_Amount_Deviation
count,50000.0,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00
mean,2023.0,6.53,15.72,3.02,4916.04,6.55,99.41,651.66,-0.00
std,0.0,3.45,8.80,2.00,89937.21,2.36,41.85,346.33,89.37
min,2023.0,1.00,1.00,0.00,0.95,1.00,0.04,0.04,-373.14
25%,2023.0,4.00,8.00,1.00,262.24,5.00,70.00,398.26,-57.05
50%,2023.0,7.00,16.00,3.00,633.71,6.00,94.18,602.92,-16.74
75%,2023.0,10.00,23.00,5.00,1675.39,8.00,121.80,845.57,39.07
max,2023.0,12.00,31.00,6.00,9784693.00,16.00,523.28,2395.02,911.61


---
## 11 · Data Quality Checks

Feature engineering can inadvertently introduce **missing values** (e.g., division by zero) or **infinite values**. We check for both.

In [21]:
# 11.1 — Missing values
missing = df.isnull().sum()
missing_cols = missing[missing > 0]

if missing_cols.empty:
    print("✅ No missing values found in any column.")
else:
    print("⚠️  Missing values detected:")
    print(missing_cols)

✅ No missing values found in any column.


In [22]:
# 11.2 — Infinite values (check only numeric columns)
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(df[numeric_cols]).sum()
inf_cols = inf_counts[inf_counts > 0]

if inf_cols.empty:
    print("✅ No infinite values found in any numeric column.")
else:
    print("⚠️  Infinite values detected:")
    print(inf_cols)

✅ No infinite values found in any numeric column.


---
## 12 · Row Count Validation

Feature engineering must **never** add or remove rows.

In [23]:
assert df.shape[0] == ORIGINAL_ROW_COUNT, (
    f"Row count mismatch! Expected {ORIGINAL_ROW_COUNT:,}, got {df.shape[0]:,}"
)
print(f"✅ Row count unchanged: {df.shape[0]:,} rows")

✅ Row count unchanged: 50,000 rows


---
## 13 · Final Dataset Preview

In [24]:
print(f"Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(5)

Final shape: 50,000 rows × 24 columns


,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Date,Account_Balance,Device_Type,Location,Merchant_Category,Previous_Fraudulent_Activity,...,Year,Month,Day,Day_of_Week,Amount_Bin,Balance_to_Amount_Ratio,User_Transaction_Count,User_Avg_Transaction_Amount,User_Total_Transaction_Amount,User_Amount_Deviation
0,TXN_33553,USER_1834,39.79,POS,2023-08-14,93213.17,Laptop,Sydney,Travel,0,...,2023,8,14,0,low,2342.628047,7,106.767143,747.37,-66.977143
1,TXN_9427,USER_7875,1.19,Bank Transfer,2023-06-07,75725.25,Mobile,New York,Clothing,0,...,2023,6,7,2,micro,63634.663866,5,11.688000,58.44,-10.498000
2,TXN_199,USER_2734,28.96,Online,2023-06-20,1588.96,Tablet,Mumbai,Restaurants,0,...,2023,6,20,1,low,54.867403,4,55.782500,223.13,-26.822500
3,TXN_12447,USER_2617,254.32,ATM Withdrawal,2023-12-07,76807.20,Tablet,New York,Clothing,0,...,2023,12,7,3,very_high,302.010066,9,161.852222,1456.67,92.467778
4,TXN_39489,USER_2014,31.28,POS,2023-11-11,92354.66,Mobile,Mumbai,Electronics,1,...,2023,11,11,5,low,2952.514706,9,71.240000,641.16,-39.960000


In [25]:
df.dtypes

Transaction_ID                              str
User_ID                                     str
Transaction_Amount                      float64
Transaction_Type                            str
Date                             datetime64[us]
Account_Balance                         float64
Device_Type                                 str
Location                                    str
Merchant_Category                           str
Previous_Fraudulent_Activity              int64
Daily_Transaction_Count                   int64
Card_Type                                   str
Card_Age                                  int64
Fraud_Label                               int64
Year                                      int32
Month                                     int32
Day                                       int32
Day_of_Week                               int32
Amount_Bin                             category
Balance_to_Amount_Ratio                 float64
User_Transaction_Count                  

---
## 14 · Save the Feature-Engineered Dataset

In [26]:
df.to_csv(OUTPUT_PATH, index=False)

file_size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print(f"✅ Saved to: {OUTPUT_PATH}")
print(f"   File size: {file_size_mb:.2f} MB")

✅ Saved to: c:\Users\sarva\Desktop\financial-fraud-detection\data\processed\feature_engineered_fraud_dataset.csv
   File size: 8.38 MB


---
## 15 · Reload & Verify the Saved File

In [27]:
df_reload = pd.read_csv(OUTPUT_PATH)

print(f"Reloaded shape: {df_reload.shape[0]:,} rows × {df_reload.shape[1]} columns")
print(f"\nColumns match original: {list(df_reload.columns) == list(df.columns)}")
print(f"Row count matches     : {df_reload.shape[0] == df.shape[0]}")

Reloaded shape: 50,000 rows × 24 columns

Columns match original: True
Row count matches     : True


In [28]:
df_reload.head(3)

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Date,Account_Balance,Device_Type,Location,Merchant_Category,Previous_Fraudulent_Activity,...,Year,Month,Day,Day_of_Week,Amount_Bin,Balance_to_Amount_Ratio,User_Transaction_Count,User_Avg_Transaction_Amount,User_Total_Transaction_Amount,User_Amount_Deviation
0,TXN_33553,USER_1834,39.79,POS,2023-08-14,93213.17,Laptop,Sydney,Travel,0,...,2023,8,14,0,low,2342.628047,7,106.767143,747.37,-66.977143
1,TXN_9427,USER_7875,1.19,Bank Transfer,2023-06-07,75725.25,Mobile,New York,Clothing,0,...,2023,6,7,2,micro,63634.663866,5,11.688000,58.44,-10.498000
2,TXN_199,USER_2734,28.96,Online,2023-06-20,1588.96,Tablet,Mumbai,Restaurants,0,...,2023,6,20,1,low,54.867403,4,55.782500,223.13,-26.822500


---
## 16 · Summary

### Features Created

| # | Feature | Source | Rationale |
|---|---------|--------|-----------|
| 1 | `Year` | `Date` | Captures year-level trend (useful if dataset spans multiple years). |
| 2 | `Month` | `Date` | Seasonal fraud patterns — e.g., holiday-season spikes. |
| 3 | `Day` | `Date` | Day-of-month patterns — e.g., salary-day activity. |
| 4 | `Day_of_Week` | `Date` | Weekend vs. weekday fraud behaviour differences. |
| 5 | `Amount_Bin` | `Transaction_Amount` | Ordinal buckets capture non-linear amount thresholds (micro-txn card testing, extreme amounts). |
| 6 | `Balance_to_Amount_Ratio` | `Account_Balance / Transaction_Amount` | Low ratio → transaction is large relative to balance → suspicious. |
| 7 | `User_Transaction_Count` | `User_ID` × `Transaction_Amount` | High-frequency transactors may have different fraud profiles. |
| 8 | `User_Avg_Transaction_Amount` | `User_ID` × `Transaction_Amount` | Establishes a per-user spending baseline. |
| 9 | `User_Total_Transaction_Amount` | `User_ID` × `Transaction_Amount` | Overall spending volume per user. |
| 10 | `User_Amount_Deviation` | `Transaction_Amount − User_Avg` | Flags transactions that deviate from the user's norm. |

### Existing Behavioral Features Retained
- `Previous_Fraudulent_Activity`
- `Daily_Transaction_Count`

### Identifiers (Not for Modelling)
- `Transaction_ID` and `User_ID` are **kept for traceability and aggregation** but must be **excluded** from model feature matrices.

### Data Leakage
- `Fraud_Label` was **not used** in any feature calculation.
- No target encoding or label-derived transformations were applied.

### What Was NOT Done (By Design)
- **No train/test split** — that belongs in the modelling notebook.
- **No SMOTE** — resampling happens after the split.
- **No scaling** — scaling is applied during model preprocessing.
- **No model training or evaluation** — out of scope for this notebook.

### Next Steps
The feature-engineered dataset is ready for **Notebook 05 — Model Building**, where we will:
1. Separate identifiers from features.
2. Perform train/test split.
3. Handle class imbalance (SMOTE).
4. Scale features.
5. Train and evaluate models.